In [ ]:
import sqlite3
import pandas as pd

: 

In [ ]:
conn = sqlite3.connect("library db")

In [ ]:
def run(query):
    return pd.read_sql_query(query, conn)

In [ ]:
library_sql = pd.read_sql_query("SELECT * FROM checkouts", conn)
library_sql.head()

In [ ]:
library_sql = pd.read_sql_query("SELECT * FROM books", conn)
library_sql.head()

In [ ]:
library_sql = pd.read_sql_query("SELECT * FROM members", conn)
library_sql.head()

In [ ]:
query = '''
SELECT checkouts.member_id,
       members.first_name,
       members.last_name,
       COUNT(checkouts.member_id) AS checkouts_count

FROM checkouts

JOIN members ON checkouts.member_id = members.member_id

GROUP BY checkouts.member_id
ORDER BY checkouts.member_id DESC
'''

run(query)

In [ ]:
query = '''
SELECT author
FROM books
WHERE author LIKE 'J%'
'''
run(query)

In [ ]:
query = '''
SELECT books.title,
       checkouts.book_id,
       COUNT(checkouts.member_id) AS checkout_count

FROM checkouts

JOIN books ON checkouts.book_id = books.book_id

GROUP BY books.book_id
ORDER BY checkout_count DESC LIMIT 5;

'''
run(query)

In [ ]:
query = '''
SELECT books.title,
       checkouts.book_id,
       members.first_name,
       members.last_name,
       members.member_id,
       COUNT(checkouts.member_id) AS checkout_count

FROM checkouts

JOIN books ON checkouts.book_id = books.book_id
JOIN members ON checkouts.member_id = members.member_id

GROUP BY books.book_id
ORDER BY checkout_count DESC LIMIT 10;

'''
run(query)

In [ ]:
query = '''
SELECT checkouts.book_id,
       members.first_name,
       members.last_name,
       members.member_id,
       members.neighborhood,
       checkouts.checkout_date,
       COUNT(checkouts.member_id) AS checkout_count

FROM checkouts

JOIN books ON checkouts.book_id = books.book_id
JOIN members ON checkouts.member_id = members.member_id

WHERE members.neighborhood = 'Maadi'

GROUP BY checkouts.checkout_date
ORDER BY checkouts.checkout_date DESC LIMIT 10 OFFSET 10;

'''
run(query)

In [ ]:
checkouts = run("SELECT * FROM checkouts")
members = run("SELECT * FROM members")

combined = pd.merge(checkouts, members, on='member_id', how='inner')

In [ ]:
combined

In [ ]:
json = pd.read_json("/content/library json")
json.head()

In [ ]:
book_catalog = pd.merge(combined, json, on='book_id', how='inner')

In [ ]:
book_catalog.head()

In [ ]:
data_unifaction = pd.read_html("/content/library html")[0]
data_unifaction.head()

In [ ]:
data_unifaction.head()

In [ ]:
#finish the first step (combine data from different places)
data_unifaction.columns = ["member_id", "book_id", "checkout_date"]
final_combined = pd.merge(book_catalog, data_unifaction, on='book_id', how='inner')
final_combined

In [ ]:
final_combined.drop("checkout_date_x", axis=1, inplace=True)
final_combined.drop("member_id_x", axis=1, inplace=True)

In [ ]:
final_combined.columns = ["checkout_id", "book_id", "return_date", "first_name", "last_name", "grade", "neighborhood", "membership_status", "join_date", "genre", "pages", "publication_year", "publisher", "member_id", "checkout_date"]
final_combined.head()

In [ ]:
final_combined.shape

In [ ]:
final_combined.head()

In [ ]:
final_combined.isna().sum()

In [ ]:
final_combined["return_date"]=final_combined["return_date"].fillna(final_combined["return_date"].mode()[0])
final_combined["grade"]=final_combined["grade"].fillna(final_combined["grade"].median())
final_combined["join_date"]=final_combined["join_date"].fillna(final_combined["join_date"].mode()[0])
final_combined["publication_year"]=final_combined["publication_year"].fillna(final_combined["publication_year"].mode()[0])

In [ ]:
final_combined.isna().sum()

In [ ]:
final_combined.duplicated().sum()

In [ ]:
final_combined[final_combined.duplicated()]

In [ ]:
final_combined=final_combined.drop_duplicates()

In [ ]:
final_combined.duplicated().sum()

In [ ]:
final_combined.to_csv("task1_combined_data.csv", index=False)

In [ ]:
final_combined.isna().sum()

In [ ]:
final_combined.isna().sum().sum()

In [ ]:
final_combined.duplicated().sum()

In [ ]:
df = pd.read_csv("/content/task1_combined_data.csv")
df

In [ ]:
df.info()

In [ ]:
df["neighborhood"].value_counts()

In [ ]:
df.shape

In [ ]:
df["neighborhood"]=df["neighborhood"].str.title().str.strip()

In [ ]:
df["neighborhood"].value_counts()

In [ ]:
df['membership_status'].value_counts()

In [ ]:
df['membership_status'] = df['membership_status'].str.title().str.strip()

In [ ]:
df['membership_status'].value_counts()

In [ ]:
df.shape

In [ ]:
inactive_df = df[df['membership_status'] == 'Inactive']
len(inactive_df)

In [ ]:
inactive_members_status_by_id = inactive_df.groupby('member_id')['membership_status'].unique()
print(inactive_members_status_by_id)


In [ ]:
df.to_csv("task2_cleaned_data.csv", index=False)

In [ ]:
df = pd.read_csv("/content/task2_cleaned_data.csv")
df.head()

In [ ]:
neighborhood_by_members = df.groupby('neighborhood')['member_id'].nunique()
print("neighborhood_by_members\n\n", neighborhood_by_members)

In [ ]:
neighborhood_by_checkouts = df.groupby('neighborhood')['checkout_id'].nunique()
print("neighborhood_by_checkouts\n\n", neighborhood_by_checkouts)